In [62]:
import boto3
import sagemaker
import joblib

In [63]:
!pip install --upgrade pip setuptools wheel

In [64]:
!pip install scikit-learn==1.4.2 --only-binary=:all:

In [65]:
#check role
role = sagemaker.get_execution_role()
print(role)

arn:aws:iam::571265126783:role/LabRole


In [66]:
#check bucket
s3 = boto3.client("s3")
response = s3.list_buckets()

for bucket in response['Buckets']:
    print(bucket['Name'])

credit-sdustink-0000
credit-sdustink-6767
initesting
sagemaker-us-east-1-571265126783
sdustink-mdmd
sdustink-mdmd123
sdustinkmdaws
sdustinkmdmodel


In [54]:
#package up a trained model to deploy it  AWS SageMaker
import tarfile

with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add(
        "model_artifact/model.joblib", 
        arcname="model.joblib"
    )

In [67]:
#Upload model to S3, because deployment will take the model form s3
s3 = boto3.client("s3")

bucket_name = "sdustink-mdmd123"

s3.upload_file(
    "model.tar.gz",
    bucket_name,
    "model/model.tar.gz"
)

In [68]:
#check model can be run or not
from inference import model_fn

model = model_fn("model_artifact")

In [69]:
import boto3
import sagemaker
from sagemaker.sklearn.model import SKLearnModel

BUCKET = "sdustink-mdmd123"
MODEL_S3_KEY = "model/model.tar.gz"
ENDPOINT_NAME = "credit-score-123"

REGION = "us-east-1"
INSTANCE_TYPE = "ml.m5.large"
FRAMEWORK_VERSION = "1.4-2"

def get_lab_role_arn() -> str:
    iam = boto3.client("iam")
    return iam.get_role(RoleName="LabRole")["Role"]["Arn"]


def main() -> None:
    boto3.setup_default_session(region_name=REGION)
    sm_session = sagemaker.Session()
    role_arn = get_lab_role_arn()
    model_s3_uri = f"s3://{BUCKET}/{MODEL_S3_KEY}"

    print(f"Role:      {role_arn}")
    print(f"Model URI: {model_s3_uri}")
    print(f"Endpoint:  {ENDPOINT_NAME}")

    model = SKLearnModel(
        model_data=model_s3_uri,
        role=role_arn,
        entry_point="inference.py",
        # source_dir=".",
        framework_version=FRAMEWORK_VERSION,
        sagemaker_session=sm_session,
    )

    print("\nDeploying endpoint (5-8 minutes)...")
    predictor = model.deploy(
        initial_instance_count=1,
        instance_type=INSTANCE_TYPE,
        endpoint_name=ENDPOINT_NAME
    )

if __name__ == "__main__":
    main()


Role:      arn:aws:iam::571265126783:role/LabRole
Model URI: s3://sdustink-mdmd123/model/model.tar.gz
Endpoint:  credit-score-123

Deploying endpoint (5-8 minutes)...
--------!

In [ ]:
#check log here: https://us-east-1.console.aws.amazon.com/cloudwatch/home?utm_source=chatgpt.com&region=us-east-1#logsV2:log-groups

In [61]:
import boto3

sm_client = boto3.client("sagemaker", region_name="us-east-1")
ENDPOINT_NAME = "credit-score-endpoint"

# 1. Delete the stuck endpoint
print(f"Deleting failed endpoint: {ENDPOINT_NAME}...")
try:
    sm_client.delete_endpoint(EndpointName=ENDPOINT_NAME)
    print("Endpoint deletion triggered.")
except Exception as e:
    print(f"No endpoint found to delete: {e}")

# 2. Delete the conflicting endpoint configuration
print(f"Deleting endpoint configuration: {ENDPOINT_NAME}...")
try:
    sm_client.delete_endpoint_config(EndpointConfigName=ENDPOINT_NAME)
    print("Endpoint configuration deletion triggered.")
except Exception as e:
    print(f"No config found to delete: {e}")

print("\nCleanup complete!")

Deleting failed endpoint: credit-score-endpoint...
Endpoint deletion triggered.
Deleting endpoint configuration: credit-score-endpoint...
Endpoint configuration deletion triggered.

Cleanup complete!
